# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and analyzing the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant) library. All schema fields, record sets, and columns are referenced by their Croissant `@id`s.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

_https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json_

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and available record sets using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
List all record sets, fields, and their Croissant `@id`s.

In [ ]:
# List all available record sets by their @id and name
print("Available record sets:")
for record_set in dataset.record_sets:
    print(f"@id: {record_set.id}	name: {getattr(record_set, 'name', '')}")

# Show the fields for each record set (by @id and name)
print("\nFields in record sets:")
for record_set in dataset.record_sets:
    print(f"\nRecord set: {record_set.id}")
    for field in record_set.fields:
        print(f"  Field @id: {field.id}\tname: {getattr(field, 'name', '')}")

## 3. Data Extraction
Load records from a chosen record set, referencing the record set and field `@id`s from the previous overview. We load each record set as a pandas DataFrame for analysis. 

In [ ]:
# List of record set @id's to extract
record_set_ids = [rs.id for rs in dataset.record_sets]

# Load all records from each record set into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set @id: {record_set_id}")

# Preview columns and head of the first record set
if record_set_ids:
    preview_id = record_set_ids[0]
    print(f"\nColumns in record set @id {preview_id}:")
    print(dataframes[preview_id].columns.tolist())
    dataframes[preview_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply typical data wrangling and analysis operations (filter, normalize, group) on fields referenced by their `@id`.

In [ ]:
import numpy as np
# Select the main record set (assume first, or replace with preferred @id)
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]
# Display all columns (i.e., field @id column names)
print(df.columns.tolist())

# Choose a numeric field by its @id (example: 'cr:field:mw', update as needed)
# You may inspect field names/ids from earlier code block.
numeric_field = None
for col in df.columns:
    if 'mw' in col.lower() or 'molecular' in col.lower() or 'weight' in col.lower():
        numeric_field = col
        break

if numeric_field is None:
    # As a fallback, use the first float or int-looking column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

if numeric_field is None:
    print("No numeric field found for analysis.")
else:
    print(f"Using numeric field: {numeric_field} (Croissant @id as column)")
    threshold = np.nanquantile(df[numeric_field], 0.95)  # Upper 5% as example
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (top 5%):")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} (z-score) for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt to group by a categorical field (e.g. 'cr:field:sample' or any plausible name)
    group_field = None
    for col in df.columns:
        if any(x in col.lower() for x in ['sample', 'group', 'type', 'category']):
            group_field = col
            break

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable group field found.")

## 5. Visualization
Visualize distributions and relationships between fields. We use Croissant `@id` column names.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} (Croissant @id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field is not None and group_field in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion

This notebook provided an overview and initial exploration of the Mass Spectrometry EV dataset using the `mlcroissant` library. All fields, record sets, and columns were addressed via their Croissant `@id`, ensuring reproducible and schema-compliant analysis. You can use this notebook as a template to further analyze and process FAIR2-compliant datasets.